In [ ]:
import os
import re
import time
import random
from pathlib import Path


try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else []

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.filters import threshold_multiotsu
from skimage.morphology import remove_small_objects, skeletonize
from skimage.measure import label, regionprops

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 120)


CONFIG = {
    "dataset_hint": "/kaggle/input/datasets/bulentsiyah/semantic-drone-dataset/dataset/semantic_drone_dataset",
    "preview_max_images": 6,
    "generated_mask_dir": "/kaggle/working/generated_masks",
    "overlay_dir": "/kaggle/working/generated_overlays",
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}



def show_progress(iterable, desc=None, total=None, leave=True, unit="item"):
    """Return a tqdm progress iterator; keeps loops readable in notebook cells."""
    return tqdm(iterable, desc=desc, total=total, leave=leave, unit=unit)


In [ ]:
def normalize_stem(path_or_stem):
    """Normalize a filename stem so image and mask names can be matched."""
    stem = Path(str(path_or_stem)).stem.lower()
    replacements = [
        "_mask", "-mask", " mask",
        "_masks", "-masks",
        "_gt", "-gt", "_groundtruth", "-groundtruth",
        "_label", "-label", "_labels", "-labels",
        "_seg", "-seg", "_segmentation", "-segmentation",
        "_binary", "-binary"
    ]
    for token in replacements:
        stem = stem.replace(token, "")
    stem = re.sub(r"[^a-z0-9]+", "", stem)
    return stem


def has_mask_indicator(path):
    """Return True if a path likely belongs to a mask/label/ground-truth directory or file."""
    p = Path(path)
    parts = [part.lower() for part in p.parts]
    name = p.stem.lower()

    # Added label_images_semantic for the Semantic Drone Dataset
    mask_tokens = {
        "mask",
        "masks",
        "label",
        "labels",
        "label_images_semantic",
        "gt",
        "groundtruth",
        "ground_truth",
        "annotations",
        "annotation"
    }

    if any(part in mask_tokens for part in parts):
        return True

    filename_tokens = [
        "_mask", "-mask",
        "mask_",
        "_gt", "-gt",
        "_label", "-label",
        "_seg", "-seg"
    ]

    return any(tok in name for tok in filename_tokens)


def candidate_dataset_roots():
    """Return the Semantic Drone Dataset root."""

    root = Path(CONFIG["dataset_hint"])

    if not root.exists():
        raise FileNotFoundError(
            f"Dataset not found:\n{root}"
        )

    return [root.resolve()]


def discover_files(root):
    """
    Load RGB images and semantic masks from the Semantic Drone Dataset.
    """

    root = Path(root)

    image_dir = root / "original_images"
    mask_dir = root / "label_images_semantic"

    if not image_dir.exists():
        raise FileNotFoundError(
            f"Image directory not found:\n{image_dir}"
        )

    if not mask_dir.exists():
        raise FileNotFoundError(
            f"Mask directory not found:\n{mask_dir}"
        )

    image_files = sorted(
        p for p in show_progress(
            image_dir.iterdir(),
            desc="Loading original images",
            leave=False,
            unit="image"
        )
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    mask_files = sorted(
        p for p in show_progress(
            mask_dir.iterdir(),
            desc="Loading semantic masks",
            leave=False,
            unit="mask"
        )
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

    return image_files, mask_files


def infer_split(path):
    parts = [part.lower() for part in Path(path).parts]

    if "train" in parts or "training" in parts:
        return "train"

    if "valid" in parts or "val" in parts or "validation" in parts:
        return "valid"

    if "test" in parts or "testing" in parts:
        return "test"

    # Semantic Drone Dataset has no train/valid/test folders
    return "all"


def pair_images_and_masks(image_files, mask_files):
    """Match images with masks using normalized stems rather than folder-specific assumptions."""
    mask_lookup = {}

    # Build a lookup table from several normalized mask-name variants.
    for m in show_progress(
        mask_files,
        desc="Indexing masks",
        leave=False,
        unit="mask"
    ):
        keys = {
            normalize_stem(m.stem),
            re.sub(r"[^a-z0-9]+", "", m.stem.lower())
        }

        for key in keys:
            if key and key not in mask_lookup:
                mask_lookup[key] = m

    rows = []

    for img in show_progress(
        image_files,
        desc="Pairing images with masks",
        leave=False,
        unit="image"
    ):
        key = normalize_stem(img.stem)
        mask = mask_lookup.get(key)

        rows.append({
            "split": infer_split(img),
            "image_path": str(img),
            "mask_path": str(mask) if mask is not None else None,
            "image_name": img.name,
            "pair_key": key,
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------------
# Main execution
# ---------------------------------------------------------------------

roots = candidate_dataset_roots()

print("Candidate roots checked:")

for r in roots[:12]:
    print("  ", r)

best_root = None
best_df = pd.DataFrame()
best_counts = (0, 0, 0)

for root in roots:

    image_files, mask_files = discover_files(root)

    if len(image_files) == 0:
        continue

    df_try = pair_images_and_masks(image_files, mask_files)

    paired_count = df_try["mask_path"].notna().sum()

    score = (
        paired_count,
        len(image_files),
        len(mask_files)
    )

    if score > best_counts:
        best_counts = score
        best_root = root
        best_df = df_try.copy()

if best_root is None or len(best_df) == 0:
    raise FileNotFoundError(
        "No image files were found. Add the Semantic Drone Dataset to the notebook, then rerun this cell."
    )

DATASET_ROOT = Path(best_root)

df_pairs = (
    best_df
    .sort_values(["split", "image_name"])
    .reset_index(drop=True)
)

print(f"\nSelected dataset root: {DATASET_ROOT}")
print(f"Discovered images: {len(df_pairs)}")
print(f"Paired masks: {df_pairs['mask_path'].notna().sum()}")
print(f"Unpaired images: {df_pairs['mask_path'].isna().sum()}")

display(df_pairs.head(10))


display(
    df_pairs.groupby("split").agg(
        images=("image_path", "count"),
        masks=("mask_path", lambda s: s.notna().sum())
    )
)

In [ ]:
# ==========================================================
# Task A - Dataset Integrity Check
# ==========================================================

integrity_results = []

print("Checking dataset integrity...\n")

for _, row in show_progress(
    df_pairs.iterrows(),
    total=len(df_pairs),
    desc="Verifying image-mask pairs",
    leave=False,
    unit="pair"
):

    image_path = row["image_path"]
    mask_path = row["mask_path"]

    status = "OK"
    image_size = None
    mask_size = None

    # ---------------------------
    # Missing mask
    # ---------------------------
    if pd.isna(mask_path):
        status = "Missing Mask"

    # ---------------------------
    # Read image
    # ---------------------------
    img = cv2.imread(image_path)

    if img is None:
        status = "Corrupted Image"

    else:
        image_size = img.shape[:2]

    # ---------------------------
    # Read mask
    # ---------------------------
    if status not in ["Missing Mask", "Corrupted Image"]:

        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

        if mask is None:
            status = "Corrupted Mask"

        else:

            mask_size = mask.shape[:2]

            if image_size != mask_size:
                status = "Size Mismatch"

    integrity_results.append({
        "image_name": row["image_name"],
        "status": status,
        "image_size": image_size,
        "mask_size": mask_size
    })

integrity_df = pd.DataFrame(integrity_results)

display(integrity_df.head())

In [ ]:
print("=" * 50)
print("DATASET INTEGRITY SUMMARY")
print("=" * 50)

display(
    integrity_df["status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Count")
)

print()

print(f"Total Images : {len(df_pairs)}")
print(f"Healthy Pairs: {(integrity_df['status']=='OK').sum()}")
print(f"Issues Found : {(integrity_df['status']!='OK').sum()}")

In [ ]:
import numpy as np
import cv2
from collections import Counter

pixel_counts = Counter()

for mask_path in show_progress(df_pairs["mask_path"].dropna(), desc="Processing masks"):

    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

    if mask is None:
        continue

    # If mask is RGB, convert to single channel
    if len(mask.shape) == 3:
        mask = mask[:, :, 0]

    unique, counts = np.unique(mask, return_counts=True)

    for u, c in zip(unique, counts):
        pixel_counts[int(u)] += int(c)



import pandas as pd

df_pixels = pd.DataFrame(
    list(pixel_counts.items()),
    columns=["class_id", "pixel_count"]
).sort_values("pixel_count", ascending=False)

df_pixels


df_pixels["percentage"] = (
    df_pixels["pixel_count"] / df_pixels["pixel_count"].sum()
) * 100

df_pixels

In [ ]:
import cv2
import numpy as np

sample_path = df_pairs["mask_path"].dropna().iloc[0]

mask = cv2.imread(sample_path, cv2.IMREAD_UNCHANGED)

print("Shape:", mask.shape)
print("dtype:", mask.dtype)

if len(mask.shape) == 2:
    print("Unique values:", np.unique(mask)[:50])

else:
    print("Unique colors (first 10):")
    print(np.unique(mask.reshape(-1, mask.shape[-1]), axis=0)[:10])

In [ ]:
# ==========================================================
# Task A - Image Size and Aspect Ratio Analysis
# ==========================================================

image_info = []

for image_path in show_progress(
    df_pairs["image_path"],
    desc="Analyzing image sizes",
    leave=False,
    unit="image"
):

    img = cv2.imread(image_path)

    if img is None:
        continue

    height, width = img.shape[:2]

    image_info.append({
        "image_name": Path(image_path).name,
        "width": width,
        "height": height,
        "aspect_ratio": width / height
    })

df_image_info = pd.DataFrame(image_info)

display(df_image_info.head())

print("=" * 50)
print("IMAGE SIZE SUMMARY")
print("=" * 50)

display(df_image_info.describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width
axes[0].hist(df_image_info["width"], bins=20)
axes[0].set_title("Image Width Distribution")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Frequency")

# Height
axes[1].hist(df_image_info["height"], bins=20)
axes[1].set_title("Image Height Distribution")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Frequency")

# Aspect Ratio
axes[2].hist(df_image_info["aspect_ratio"], bins=20)
axes[2].set_title("Aspect Ratio Distribution")
axes[2].set_xlabel("Aspect Ratio (W/H)")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# Task A - Find Representative Image for Each Class
# ==========================================================

representative_samples = []

# All observed classes from previous EDA
observed_classes = sorted(df_pixels["class_id"].tolist())

for class_id in observed_classes:

    found = False

    for _, row in df_pairs.iterrows():

        if pd.isna(row["mask_path"]):
            continue

        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

        if mask is None:
            continue

        # Check whether this class exists in the mask
        if np.any(mask == class_id):

            representative_samples.append({
                "class_id": class_id,
                "image_path": row["image_path"],
                "mask_path": row["mask_path"]
            })

            found = True
            break

    if not found:
        print(f"Class {class_id} not found.")

representative_df = pd.DataFrame(representative_samples)

display(representative_df)

In [ ]:
# ==========================================================
# Task A - Display Representative Image–Mask Pairs
# ==========================================================

for _, row in representative_df.iterrows():

    image = cv2.imread(row["image_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

    plt.figure(figsize=(12,5))

    # Original image
    plt.subplot(1,2,1)
    plt.imshow(image)
    plt.title(f"Original Image\nClass {row['class_id']}")
    plt.axis("off")

    # Mask
    plt.subplot(1,2,2)
    plt.imshow(mask, cmap="tab20")
    plt.title("Segmentation Mask")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================================
# Task A - Representative Image-Mask Pairs for Every Class
# ==========================================================

for _, row in representative_df.iterrows():

    class_id = row["class_id"]

    # Read original image
    image = cv2.imread(row["image_path"])
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Read semantic mask
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)

    # Binary mask of only the selected class
    class_mask = (mask == class_id)

    plt.figure(figsize=(18,6))

    # -------------------------------------------------
    # Original Image
    # -------------------------------------------------
    plt.subplot(1,3,1)
    plt.imshow(image)
    plt.title(f"Original Image\nClass {class_id}")
    plt.axis("off")

    # -------------------------------------------------
    # Complete Semantic Mask
    # -------------------------------------------------
    plt.subplot(1,3,2)
    plt.imshow(mask, cmap="tab20")
    plt.title("Complete Semantic Mask")
    plt.axis("off")

    # -------------------------------------------------
    # Selected Class Only
    # -------------------------------------------------
    plt.subplot(1,3,3)
    plt.imshow(class_mask, cmap="gray")
    plt.title(f"Pixels of Class {class_id}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()